#E23CSEU0868
#Lab-7
#B-29

In [1]:
import sklearn
import scikeras

print("Scikit-learn:", sklearn.__version__)
print("SciKeras:", scikeras.__version__)

Scikit-learn: 1.5.2
SciKeras: 0.13.0


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mdssh784/pima-indians-diabetes-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'pima-indians-diabetes-dataset' dataset.
Path to dataset files: /kaggle/input/pima-indians-diabetes-dataset


In [3]:
import os

print("Files in dataset folder:")
print(os.listdir(path))

Files in dataset folder:
['diabetes.csv']


In [4]:
import pandas as pd

data = pd.read_csv(os.path.join(path, "diabetes.csv"))

print("Dataset Shape:", data.shape)
data.head()

Dataset Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
print("Column Names:")
print(data.columns)

print("\nDataset Information:")
data.info()

print("\nMissing Values:")
print(data.isnull().sum())

Column Names:
Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Missing Values:
Pregnancies               

In [6]:
X = data.drop("Outcome", axis=1)
y = data["Outcome"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (768, 8)
y shape: (768,)


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (614, 8)
Testing data: (154, 8)


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Data preprocessing completed.")

Data preprocessing completed.


In [9]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [10]:
def create_model(
    learning_rate=0.001,
    init_mode="uniform",
    activation="relu",
    dropout_rate=0.0,
    neurons=16
):

    model = Sequential()

    model.add(
        Dense(
            neurons,
            input_dim=X_train.shape[1],
            kernel_initializer=init_mode,
            activation=activation
        )
    )

    model.add(Dropout(dropout_rate))

    model.add(
        Dense(
            1,
            kernel_initializer=init_mode,
            activation="sigmoid"
        )
    )

    optimizer = Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [11]:
model = KerasClassifier(
    model=create_model,
    verbose=0
)

In [12]:
param_grid = {
    "batch_size": [10, 20, 40],
    "epochs": [50, 100]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_result = grid.fit(X_train, y_train)

print("Best Accuracy:", grid_result.best_score_)
print("Best Parameters:", grid_result.best_params_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Accuracy: 0.7736888251235454
Best Parameters: {'batch_size': 10, 'epochs': 100}


In [13]:
param_grid = {
    "model__learning_rate": [0.001, 0.01, 0.1]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_result = grid.fit(X_train, y_train)

print("Best Accuracy:", grid_result.best_score_)
print("Best Learning Rate:", grid_result.best_params_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Accuracy: 0.7606169296987088
Best Learning Rate: {'model__learning_rate': 0.1}


In [14]:
param_grid = {
    "model__init_mode": [
        "uniform",
        "normal",
        "glorot_uniform",
        "he_uniform"
    ]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_result = grid.fit(X_train, y_train)

print("Best Accuracy:", grid_result.best_score_)
print("Best Weight Initialization:", grid_result.best_params_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Accuracy: 0.6774988043998086
Best Weight Initialization: {'model__init_mode': 'uniform'}


In [15]:
param_grid = {
    "model__activation": [
        "relu",
        "tanh",
        "sigmoid"
    ]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_result = grid.fit(X_train, y_train)

print("Best Accuracy:", grid_result.best_score_)
print("Best Activation Function:", grid_result.best_params_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Accuracy: 0.7084568786864338
Best Activation Function: {'model__activation': 'tanh'}


In [16]:
param_grid = {
    "model__dropout_rate": [0.0, 0.2, 0.3, 0.5]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_result = grid.fit(X_train, y_train)

print("Best Accuracy:", grid_result.best_score_)
print("Best Dropout Rate:", grid_result.best_params_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Accuracy: 0.6710266220309262
Best Dropout Rate: {'model__dropout_rate': 0.3}


In [17]:
param_grid = {
    "model__neurons": [8, 16, 32, 64]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_result = grid.fit(X_train, y_train)

print("Best Accuracy:", grid_result.best_score_)
print("Best Number of Neurons:", grid_result.best_params_)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Accuracy: 0.7181731229076996
Best Number of Neurons: {'model__neurons': 32}


In [18]:
best_model = grid_result.best_estimator_

test_accuracy = best_model.score(X_test, y_test)

print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.6753246753246753


In [19]:
y_pred = best_model.predict(X_test)

print("First 20 Predictions:")
print(y_pred[:20])

First 20 Predictions:
[0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
